# Construccion `market_cap_t = close_t * shares_outstanding_t`

**Matiz:**

- el file seguirá llamándose population_target_pti.parquet
- lo que cambia es el RUN_ID y por tanto la carpeta:
    - population_target_pti_1B

Eso es lo correcto.
Luego, si quieres, sobre ese run nuevo hacemos:

- market_cap_cutoff_lt_2b...
- market_cap_cutoff_lt_1b...

**NOTA** : `TTL_DAYS = 180`

Para no arrastrar shares demasiado viejas y clasificar mal el market cap.

  Sin TTL, podrías hacer esto:

  - usar unas shares de hace 2 años
  - multiplicarlas por el precio actual
  - y fingir un market_cap_t PTI que ya no es creíble

TTL_DAYS = 180 es una regla de frescura:

- las shares observadas por filing solo viven 180 días
- después caducan
- y ya no se usan para calcular market_cap_t

In [58]:
from pathlib import Path

SCRIPT = Path(
    r"C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\build_population_target_pti.py"
)

UNIVERSE_PTI_ROOT = Path(
    r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_panel_pti"
)

OHLCV_DAILY_ROOT = Path(
    r"D:\ohlcv_daily"
)

INCOME_STATEMENTS_ROOT = Path(
    r"D:\financial\income_statements"
)

BALANCE_SHEETS_ROOT = Path(
    r"D:\financial\balance_sheets"
)

RUN_ID = "population_target_pti_1B"
OUT_BASE = Path(
    r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\population_target_pti"
)
OUT_DIR = OUT_BASE / RUN_ID
DATE_TO = "2026-12-31"
TTL_DAYS = 180
MAX_TICKERS = None

print("SCRIPT:", SCRIPT)
print("UNIVERSE_PTI_ROOT:", UNIVERSE_PTI_ROOT, "exists=", UNIVERSE_PTI_ROOT.exists())
print("OHLCV_DAILY_ROOT:", OHLCV_DAILY_ROOT, "exists=", OHLCV_DAILY_ROOT.exists())
print("INCOME_STATEMENTS_ROOT:", INCOME_STATEMENTS_ROOT, "exists=", INCOME_STATEMENTS_ROOT.exists())
print("BALANCE_SHEETS_ROOT:", BALANCE_SHEETS_ROOT, "exists=", BALANCE_SHEETS_ROOT.exists())
print("OUT_DIR:", OUT_DIR)
print("RUN_ID:", RUN_ID)

exec(SCRIPT.read_text(encoding="utf-8"), globals(), globals())

SCRIPT: C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\build_population_target_pti.py
UNIVERSE_PTI_ROOT: C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_panel_pti exists= False
OHLCV_DAILY_ROOT: D:\ohlcv_daily exists= True
INCOME_STATEMENTS_ROOT: D:\financial\income_statements exists= True
BALANCE_SHEETS_ROOT: D:\financial\balance_sheets exists= True
OUT_DIR: C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\population_target_pti\population_target_pti_1B
RUN_ID: population_target_pti_1B


FileNotFoundError: No existe path requerido: C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_panel_pti

In [56]:
import subprocess
from pathlib import Path

PYTHON_EXE = Path(
    r"C:\TSIS_Data\v1\backtest_SmallCaps\backtest\Scripts\python.exe"
)

SCRIPT = Path(
    r"C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\build_population_target_pti.py"
)

cmd = [
    str(PYTHON_EXE),
    "-c",
    (
        "from pathlib import Path; "
        f"SCRIPT=Path(r'{SCRIPT}'); "
        r"UNIVERSE_PTI_ROOT=Path(r'C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_panel_pti');"
        r"OHLCV_DAILY_ROOT=Path(r'D:\ohlcv_daily'); "
        r"INCOME_STATEMENTS_ROOT=Path(r'D:\financial\income_statements'); "
        r"BALANCE_SHEETS_ROOT=Path(r'D:\financial\balance_sheets'); "
        r"RUN_ID='population_target_pti_1B'; "
        r"DATE_FROM='2005-01-01'; "
        r"DATE_TO='2026-12-31'; "
        r"TTL_DAYS=180; "
        r"MAX_TICKERS=None; "
        "exec(SCRIPT.read_text(encoding='utf-8'), globals(), globals())"
    ),
]

res = subprocess.run(cmd, capture_output=True, text=True)

print("returncode:", res.returncode)
print("\n=== STDOUT ===\n")
print(res.stdout)
print("\n=== STDERR ===\n")
print(res.stderr)

returncode: 1

=== STDOUT ===



=== STDERR ===

Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "<string>", line 7, in <module>
ModuleNotFoundError: No module named 'duckdb'



# intento de reconstruir desde local `tickers_panel_pti`

Luego comararé la descarga, con esta, con la que ya hay en   
`C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\population_target_pti\population_target_pti_run_01\population_target_pti.parquet`

Usa esta celda. Construye un tickers_panel_pti local aproximado desde:

- D:\reference\all_tickers
- D:\reference\overview

y lo deja en un outdir nuevo para luego compararlo con el histórico/rebuild.

## Qué hace exactamente

- usa D:\reference\all_tickers como spine base
- usa D:\reference\overview para traer list_date
- deriva:
    - entity_id
    - exchange_priority
    - has_composite_figi
    - has_share_class_figi
    - has_list_date
- rellena delisted_utc = null
- guarda un tickers_panel_pti local aproximado en:
- C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti_from_reference_compare\tickers_panel_pti

## Nota

No es bit a bit igual al histórico, pero sí un candidato serio para comparar y validar.

Después te doy la celda de comparación contra:

- el rebuild remoto que está corriendo
- o contra el schema esperado histórico.

In [60]:
from pathlib import Path
import pandas as pd

ALL_TICKERS_ROOT = Path(r"D:\reference\all_tickers")
OVERVIEW_ROOT = Path(r"D:\reference\overview")

OUT_ROOT = Path(
    r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti_from_reference_compare"
)
OUT_PANEL = OUT_ROOT / "tickers_panel_pti"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
OUT_PANEL.mkdir(parents=True, exist_ok=True)

# 1) all_tickers como base del panel diario
all_files = sorted(ALL_TICKERS_ROOT.rglob("*.parquet"))
print("all_tickers_files:", len(all_files))

frames = []
for p in all_files:
    df = pd.read_parquet(p).copy()
    frames.append(df)

panel = pd.concat(frames, ignore_index=True)
panel["ticker"] = panel["ticker"].astype(str).str.upper().str.strip()
panel["primary_exchange"] = panel["primary_exchange"].astype(str).str.upper().str.strip()
panel["snapshot_date"] = pd.to_datetime(panel["snapshot_date"], errors="coerce")

# 2) overview para recuperar list_date por ticker
ov_files = sorted(OVERVIEW_ROOT.rglob("*.parquet"))
print("overview_files:", len(ov_files))

ov_frames = []
for p in ov_files:
    try:
        df = pd.read_parquet(
            p,
            columns=["ticker", "list_date", "cik", "composite_figi", "share_class_figi"]
        ).copy()
        ov_frames.append(df)
    except Exception:
        pass

overview = pd.concat(ov_frames, ignore_index=True) if ov_frames else pd.DataFrame(
    columns=["ticker", "list_date", "cik", "composite_figi", "share_class_figi"]
)

overview["ticker"] = overview["ticker"].astype(str).str.upper().str.strip()
overview["list_date"] = pd.to_datetime(overview["list_date"], errors="coerce")

# prioridad simple: primera list_date no nula por ticker
overview_best = (
    overview.sort_values(["ticker", "list_date"], na_position="last")
    .drop_duplicates(subset=["ticker"], keep="first")
    .copy()
)

panel = panel.merge(
    overview_best[["ticker", "list_date"]],
    on="ticker",
    how="left",
)

# 3) derivadas para parecerse a tickers_panel_pti
exchange_priority_map = {"XNAS": 0, "XNYS": 1, "XASE": 2, "ARCX": 3}

panel["entity_id"] = (
    panel["composite_figi"]
    .fillna(panel["share_class_figi"])
    .fillna(panel["ticker"] + "|" + panel["primary_exchange"].fillna("UNK"))
)

panel["exchange_priority"] = (
    panel["primary_exchange"].map(exchange_priority_map).fillna(999).astype(int)
)

panel["has_composite_figi"] = panel["composite_figi"].notna().astype(int)
panel["has_share_class_figi"] = panel["share_class_figi"].notna().astype(int)
panel["has_list_date"] = panel["list_date"].notna().astype(int)

if "delisted_utc" not in panel.columns:
    panel["delisted_utc"] = pd.NaT

# 4) dejar columnas en el orden esperado
expected_cols = [
    "ticker",
    "name",
    "market",
    "locale",
    "primary_exchange",
    "type",
    "active",
    "currency_name",
    "cik",
    "composite_figi",
    "share_class_figi",
    "list_date",
    "delisted_utc",
    "last_updated_utc",
    "snapshot_date",
    "entity_id",
    "exchange_priority",
    "has_composite_figi",
    "has_share_class_figi",
    "has_list_date",
]

for c in expected_cols:
    if c not in panel.columns:
        panel[c] = pd.NA

panel = panel[expected_cols].copy()
panel = panel.sort_values(["snapshot_date", "ticker", "exchange_priority"]).reset_index(drop=True)

# 5) guardar particionado por año/mes, parecido al panel original
panel["snapshot_year"] = panel["snapshot_date"].dt.year
panel["snapshot_month"] = panel["snapshot_date"].dt.month

for (yy, mm), part in panel.groupby(["snapshot_year", "snapshot_month"], dropna=False):
      if pd.isna(yy) or pd.isna(mm):
          continue
      out_dir = OUT_PANEL / str(int(yy)) / str(int(mm))
      out_dir.mkdir(parents=True, exist_ok=True)
      date_min = part["snapshot_date"].min().date()
      date_max = part["snapshot_date"].max().date()
      out_file = out_dir / f"snapshot_date={date_min}_to_{date_max}.parquet"
      part[expected_cols].to_parquet(out_file, index=False)

summary = {
    "out_root": str(OUT_ROOT),
    "out_panel": str(OUT_PANEL),
    "rows_total": int(len(panel)),
    "tickers_unique": int(panel["ticker"].nunique()),
    "snapshot_date_min": str(panel["snapshot_date"].min()),
    "snapshot_date_max": str(panel["snapshot_date"].max()),
    "with_list_date": int(panel["list_date"].notna().sum()),
    "without_list_date": int(panel["list_date"].isna().sum()),
}

print(summary)
display(panel.head(10).T)

all_tickers_files: 3109
overview_files: 12468
{'out_root': 'C:\\TSIS_Data\\v1\\backtest_SmallCaps\\data\\reference\\universe_pti_from_reference_compare', 'out_panel': 'C:\\TSIS_Data\\v1\\backtest_SmallCaps\\data\\reference\\universe_pti_from_reference_compare\\tickers_panel_pti', 'rows_total': 12977501, 'tickers_unique': 13124, 'snapshot_date_min': '2005-01-02 00:00:00', 'snapshot_date_max': '2026-03-09 00:00:00', 'with_list_date': 8915597, 'without_list_date': 4061904}


,0,1,2,3,4,5,6,7,8,9
ticker,A,AA,AAA,AAI,AAP,ABB,ABC,ABG,ABI,ABK
name,"AGILENT TECHNOLOGIES, INC",ALCOA INC,ALTANA AKTIENGESELLSCHAFT SPON ADR,AIRTRAN HOLDINGS INC,ADVANCE AUTO PARTS INC,ABB LTD ADS ( 1 REG SHS),AMERISOURCEBERGEN CORP,ASBURY AUTOMOTIVE GROUP INC.,APPLERA CP-APPLIED BIOSYSTEMS,AMBAC FINANCIAL GROUP INC.
market,stocks,stocks,stocks,stocks,stocks,stocks,stocks,stocks,stocks,stocks
locale,us,us,us,us,us,us,us,us,us,us
primary_exchange,XNYS,XNYS,XNYS,XNYS,XNYS,XNYS,XNYS,XNYS,XNYS,XNYS
type,CS,CS,CS,CS,CS,CS,CS,CS,CS,CS
active,True,True,True,True,True,True,True,True,True,True
currency_name,usd,usd,usd,usd,usd,usd,usd,usd,usd,usd
cik,0001090872,0000004281,NaN,0000948846,0001158449,0001091587,0001140859,0001144980,0000719545,0000874501
composite_figi,BBG000C2V3D6,NaN,NaN,NaN,BBG000F7RCJ1,BBG000DK5Q25,BBG000MDCQC2,BBG000BKDWB5,NaN,NaN


# 00 · Corte 2B del universo sin tocar raws Polygon

## Objetivo

Construir un artefacto derivado, reproducible y auditable para clasificar el universo de:

- `tickers_2005_2026_upper.parquet`

en función de su `market_cap_t` en el último día observado, distinguiendo:

- activas `< 2B`
- inactivas que murieron `< 2B`
- `>= 2B`
- no clasificables por falta de `market_cap_t`

## Regla de negocio

Para cada ticker:

- `close_t` sale de `D:\ohlcv_daily`
- `shares_outstanding_t` sale de `D:\financial\income_statements`
- `market_cap_t = close_t * shares_outstanding_t`
- no se usan datos futuros
- para inactivas, el “momento de muerte” se aproxima con el último día observado en `ohlcv_daily`

## Importante

Este notebook:

- no modifica raws de Polygon
- no modifica `D:\ohlcv_daily`
- no modifica `D:\financial`
- solo genera artefactos derivados

In [30]:
from pathlib import Path
import pandas as pd

pti_path = Path(
    r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\population_target_pti\population_target_pti_run_01\population_target_pti.parquet"
)

pti = pd.read_parquet(pti_path, columns=["ticker", "date", "market_cap_t"]).copy()
pti["ticker"] = pti["ticker"].astype(str).str.upper().str.strip()
pti["date"] = pd.to_datetime(pti["date"], errors="coerce")
pti["market_cap_t"] = pd.to_numeric(pti["market_cap_t"], errors="coerce")

tickers_total = pti["ticker"].dropna().nunique()
tickers_with_mc = pti.loc[pti["market_cap_t"].notna(), "ticker"].dropna().nunique()
tickers_without_mc = tickers_total - tickers_with_mc

last_classifiable = (
    pti.loc[pti["market_cap_t"].notna()]
        .sort_values(["ticker", "date"])
        .drop_duplicates(subset=["ticker"], keep="last")
        .copy()
)

print("pti_path:", pti_path)
print()
print("tickers_total_in_pti:", tickers_total)
print("tickers_with_market_cap_t:", tickers_with_mc)
print("tickers_without_market_cap_t:", tickers_without_mc)
print("lt_2b_last_classifiable:", int((last_classifiable["market_cap_t"] < 2_000_000_000).sum()))
print("ge_2b_last_classifiable:", int((last_classifiable["market_cap_t"] >= 2_000_000_000).sum()))
print()

print("sample_with_market_cap:")
display(
    last_classifiable[["ticker", "date", "market_cap_t"]]
    .sort_values(["ticker"])
    .head(2)
)

pti_path: C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\population_target_pti\population_target_pti_run_01\population_target_pti.parquet

tickers_total_in_pti: 13066
tickers_with_market_cap_t: 7661
tickers_without_market_cap_t: 5405
lt_2b_last_classifiable: 5478
ge_2b_last_classifiable: 2183

sample_with_market_cap:


,ticker,date,market_cap_t
7192571,A,2026-03-06,3.267988e+10
22307543,AA,2026-03-06,1.559045e+10


In [ ]:
# Celda 3: validación de inputs
from pathlib import Path
from datetime import datetime
import json

import pandas as pd
import pyarrow.parquet as pq

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 2000)
pd.set_option("display.max_colwidth", 200)

# 
UNIVERSE_PARQUET = Path(
    r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_2005_2026_upper.parquet"
)

POPULATION_TARGET_PTI = Path(
    r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\population_target_pti\population_target_pti_run_01\population_target_pti.parquet"
)

OFFICIAL_LIFECYCLE_CSV = Path(
    r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\official_lifecycle_compiled.csv"
)

RUN_ID = "20260320_market_cap_last_observed_cutoff"
OUT_DIR = Path(
    rf"C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\market_cap_last_observed_cutoff\{RUN_ID}"
)

DETAIL_PARQUET = OUT_DIR / "market_cap_last_observed_by_ticker.parquet"
DETAIL_CSV = OUT_DIR / "market_cap_last_observed_by_ticker.csv"
CUT_PARQUET = OUT_DIR / "market_cap_cutoff_lt_2b_active_inactive.parquet"
CUT_CSV = OUT_DIR / "market_cap_cutoff_lt_2b_active_inactive.csv"
SUMMARY_JSON = OUT_DIR / "market_cap_cutoff_lt_2b_summary.json"

print("UNIVERSE_PARQUET:", UNIVERSE_PARQUET)
print("POPULATION_TARGET_PTI:", POPULATION_TARGET_PTI)
print("OFFICIAL_LIFECYCLE_CSV:", OFFICIAL_LIFECYCLE_CSV)
print("OUT_DIR:", OUT_DIR)

UNIVERSE_PARQUET: C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_2005_2026_upper.parquet
POPULATION_TARGET_PTI: C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\population_target_pti\population_target_pti_run_01\population_target_pti.parquet
OFFICIAL_LIFECYCLE_CSV: C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\official_lifecycle_compiled.csv
OUT_DIR: C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\market_cap_last_observed_cutoff\20260320_market_cap_last_observed_cutoff


In [ ]:
# lo cruzo con universo base

required_paths = [
    UNIVERSE_PARQUET,
    POPULATION_TARGET_PTI,
    OFFICIAL_LIFECYCLE_CSV,
]

for p in required_paths:
    print(f"{p} -> exists={p.exists()}")

u = pd.read_parquet(UNIVERSE_PARQUET, columns=["ticker"]).copy()
u["ticker"] = u["ticker"].astype(str).str.upper().str.strip()
u = u.dropna().drop_duplicates().sort_values("ticker").reset_index(drop=True)

print()
print("universe_tickers:", len(u))
display(u.head(10))

C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_2005_2026_upper.parquet -> exists=True
C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\population_target_pti\population_target_pti_run_01\population_target_pti.parquet -> exists=True
C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\official_lifecycle_compiled.csv -> exists=True

universe_tickers: 12468


,ticker
0,A
1,AA
2,AAA
3,AABA
4,AAC
5,AACB
6,AACI
7,AACQ
8,AACT
9,AAGR


In [3]:
# Celda 4: cargar PTI y construir el último estado observable por ticker

cols_needed = [
    "date",
    "ticker",
    "status",
    "close_t",
    "shares_outstanding_t",
    "shares_source",
    "shares_observed_date",
    "shares_period_end",
    "shares_age_days",
    "market_cap_t",
    "is_small_cap_t",
]

pti = pd.read_parquet(POPULATION_TARGET_PTI, columns=cols_needed).copy()

pti["ticker"] = pti["ticker"].astype(str).str.upper().str.strip()
pti["date"] = pd.to_datetime(pti["date"], errors="coerce")
pti["shares_observed_date"] = pd.to_datetime(pti["shares_observed_date"], errors="coerce")
pti["shares_period_end"] = pd.to_datetime(pti["shares_period_end"], errors="coerce")

pti = pti[pti["ticker"].isin(set(u["ticker"]))].copy()

print("pti_rows:", len(pti))
print("pti_unique_tickers:", pti["ticker"].nunique())
print()

# primer y último día observado por ticker
seen = (
    pti.groupby("ticker", as_index=False)
        .agg(
            first_seen_date=("date", "min"),
            last_observed_date=("date", "max"),
        )
)

# última fila observada por ticker
last_obs = (
    pti.sort_values(["ticker", "date"])
        .drop_duplicates(subset=["ticker"], keep="last")
        .copy()
)

# última fila clasificable por ticker (market_cap_t no nulo)
last_classifiable = (
    pti.loc[pti["market_cap_t"].notna()]
        .sort_values(["ticker", "date"])
        .drop_duplicates(subset=["ticker"], keep="last")
        .copy()
)

last_obs = last_obs.rename(columns={
    "date": "last_row_date",
    "status": "status_last_row",
    "close_t": "close_t_last_row",
    "shares_outstanding_t": "shares_outstanding_t_last_row",
    "shares_source": "shares_source_last_row",
    "shares_observed_date": "shares_observed_date_last_row",
    "shares_period_end": "shares_period_end_last_row",
    "shares_age_days": "shares_age_days_last_row",
    "market_cap_t": "market_cap_t_last_row",
    "is_small_cap_t": "is_small_cap_t_last_row",
})

last_classifiable = last_classifiable.rename(columns={
    "date": "anchor_date_used",
    "status": "status_at_anchor",
    "close_t": "close_t",
    "shares_outstanding_t": "shares_outstanding_t",
    "shares_source": "shares_source",
    "shares_observed_date": "shares_observed_date",
    "shares_period_end": "shares_period_end",
    "shares_age_days": "shares_age_days",
    "market_cap_t": "market_cap_t",
    "is_small_cap_t": "is_small_cap_t",
})

base = u.merge(seen, on="ticker", how="left")
base = base.merge(
    last_obs[
        [
            "ticker",
            "last_row_date",
            "status_last_row",
            "close_t_last_row",
            "shares_outstanding_t_last_row",
            "shares_source_last_row",
            "shares_observed_date_last_row",
            "shares_period_end_last_row",
            "shares_age_days_last_row",
            "market_cap_t_last_row",
            "is_small_cap_t_last_row",
        ]
    ],
    on="ticker",
    how="left",
)
base = base.merge(
    last_classifiable[
        [
            "ticker",
            "anchor_date_used",
            "status_at_anchor",
            "close_t",
            "shares_outstanding_t",
            "shares_source",
            "shares_observed_date",
            "shares_period_end",
            "shares_age_days",
            "market_cap_t",
            "is_small_cap_t",
        ]
    ],
    on="ticker",
    how="left",
)

print("base_rows:", len(base))
print("base_unique_tickers:", base["ticker"].nunique())
print()

display(base.head(10).T)

pti_rows: 29072268
pti_unique_tickers: 12468

base_rows: 12468
base_unique_tickers: 12468



,0,1,2,3,4,5,6,7,8,9
ticker,A,AA,AAA,AABA,AAC,AACB,AACI,AACQ,AACT,AAGR
first_seen_date,2005-01-01 00:00:00,2005-01-01 00:00:00,2005-01-01 00:00:00,2017-06-19 00:00:00,2008-10-03 00:00:00,2025-04-07 00:00:00,2021-11-10 00:00:00,2020-09-04 00:00:00,2023-06-12 00:00:00,2023-12-07 00:00:00
last_observed_date,2026-03-09 00:00:00,2026-03-09 00:00:00,2007-05-21 00:00:00,2019-10-06 00:00:00,2023-11-06 00:00:00,2026-03-09 00:00:00,2025-10-29 00:00:00,2021-06-24 00:00:00,2025-09-24 00:00:00,2024-09-25 00:00:00
last_row_date,2026-03-09 00:00:00,2026-03-09 00:00:00,2007-05-21 00:00:00,2019-10-06 00:00:00,2023-11-06 00:00:00,2026-03-09 00:00:00,2025-10-29 00:00:00,2021-06-24 00:00:00,2025-09-24 00:00:00,2024-09-25 00:00:00
status_last_row,active,active,active,active,active,active,active,active,active,active
close_t_last_row,NaN,NaN,NaN,NaN,10.79,NaN,10.29,9.99,9.49,0.1326
shares_outstanding_t_last_row,284000000.0,261365384.5,NaN,NaN,NaN,NaN,NaN,NaN,55470890.5,28394669.25
shares_source_last_row,diluted,diluted,NaN,NaN,diluted,NaN,NaN,NaN,diluted,diluted
shares_observed_date_last_row,2026-03-03 00:00:00,2026-02-26 00:00:00,NaT,NaT,2023-03-10 00:00:00,NaT,NaT,NaT,2025-08-12 00:00:00,2024-05-20 00:00:00
shares_period_end_last_row,2026-01-31 00:00:00,2025-12-31 00:00:00,NaT,NaT,2021-12-31 00:00:00,NaT,NaT,NaT,2024-06-30 00:00:00,2024-03-31 00:00:00


In [4]:
# Celda 5: cobertura de market_cap_t recalculado

summary = {
    "universe_tickers": int(base["ticker"].nunique()),
    "with_first_seen_date": int(base["first_seen_date"].notna().sum()),
    "with_last_observed_date": int(base["last_observed_date"].notna().sum()),
    "with_last_row": int(base["last_row_date"].notna().sum()),
    "with_anchor_date_used": int(base["anchor_date_used"].notna().sum()),
    "with_market_cap_t_recomputed": int(base["market_cap_t"].notna().sum()),
    "without_market_cap_t_recomputed": int(base["market_cap_t"].isna().sum()),
    "lt_2b_recomputed": int((base["market_cap_t"] < 2_000_000_000).sum()),
    "ge_2b_recomputed": int((base["market_cap_t"] >= 2_000_000_000).sum()),
}

print(json.dumps(summary, indent=2, ensure_ascii=False))
print()

if "market_cap_t" in base.columns:
    mc = pd.to_numeric(base["market_cap_t"], errors="coerce") / 1_000_000
    print("market_cap_t_mm_describe:")
    print(mc.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).to_string())
    print()

print("sample_without_recomputed_market_cap:")
display(
    base.loc[base["market_cap_t"].isna(), [
        "ticker",
        "first_seen_date",
        "last_observed_date",
        "last_row_date",
        "status_last_row",
        "close_t_last_row",
        "shares_outstanding_t_last_row",
        "market_cap_t_last_row",
        "anchor_date_used",
        "market_cap_t",
    ]].head(20)
)

print("sample_with_recomputed_market_cap:")
display(
    base.loc[base["market_cap_t"].notna(), [
        "ticker",
        "anchor_date_used",
        "status_at_anchor",
        "close_t",
        "shares_outstanding_t",
        "shares_source",
        "shares_observed_date",
        "shares_age_days",
        "market_cap_t",
        "is_small_cap_t",
    ]].head(20)
)

{
  "universe_tickers": 12468,
  "with_first_seen_date": 12468,
  "with_last_observed_date": 12468,
  "with_last_row": 12468,
  "with_anchor_date_used": 7661,
  "with_market_cap_t_recomputed": 7661,
  "without_market_cap_t_recomputed": 4807,
  "lt_2b_recomputed": 5478,
  "ge_2b_recomputed": 2183
}

market_cap_t_mm_describe:
count    7.661000e+03
mean     4.216778e+10
std      3.690806e+12
min      7.497970e-08
1%       2.314545e-02
5%       1.271778e+00
25%      4.759837e+01
50%      3.795000e+02
75%      2.642553e+03
95%      3.380886e+04
99%      1.706762e+05
max      3.230457e+14

sample_without_recomputed_market_cap:


,ticker,first_seen_date,last_observed_date,last_row_date,status_last_row,close_t_last_row,shares_outstanding_t_last_row,market_cap_t_last_row,anchor_date_used,market_cap_t
2,AAA,2005-01-01,2007-05-21,2007-05-21,active,NaN,NaN,NaN,NaT,NaN
3,AABA,2017-06-19,2019-10-06,2019-10-06,active,NaN,NaN,NaN,NaT,NaN
5,AACB,2025-04-07,2026-03-09,2026-03-09,active,NaN,NaN,NaN,NaT,NaN
6,AACI,2021-11-10,2025-10-29,2025-10-29,active,10.2900,NaN,NaN,NaT,NaN
7,AACQ,2020-09-04,2021-06-24,2021-06-24,active,9.9900,NaN,NaN,NaT,NaN
10,AAI,2005-01-01,2011-05-02,2011-05-02,active,7.4300,NaN,NaN,NaT,NaN
13,AAM,2024-09-16,2026-02-02,2026-02-02,active,10.6600,NaN,NaN,NaT,NaN
21,AAPC,2016-10-25,2017-10-04,2017-10-04,active,10.6000,NaN,NaN,NaT,NaN
27,AAU,2008-10-03,2024-04-04,2024-04-04,active,0.1465,NaN,NaN,NaT,NaN
28,AAUC,2025-06-09,2026-03-09,2026-03-09,active,NaN,NaN,NaN,NaT,NaN


sample_with_recomputed_market_cap:


,ticker,anchor_date_used,status_at_anchor,close_t,shares_outstanding_t,shares_source,shares_observed_date,shares_age_days,market_cap_t,is_small_cap_t
0,A,2026-03-06,active,115.0700,2.840000e+08,diluted,2026-03-03,3.0,3.267988e+10,False
1,AA,2026-03-06,active,59.6500,2.613654e+08,diluted,2026-02-26,8.0,1.559045e+10,False
4,AAC,2023-09-06,active,10.6950,9.375000e+07,diluted,2023-03-10,180.0,1.002656e+09,True
8,AACT,2025-09-24,active,9.4900,5.547089e+07,diluted,2025-08-12,43.0,5.264188e+08,True
9,AAGR,2024-09-25,active,0.1326,2.839467e+07,diluted,2024-05-20,128.0,3.765133e+06,True
11,AAIC,2023-12-13,active,4.8400,2.924300e+07,diluted,2023-11-14,29.0,1.415361e+08,True
12,AAL,2026-03-06,active,11.1800,6.887182e+08,diluted,2026-02-18,16.0,7.699870e+09,False
14,AAMC,2024-09-16,active,1.2500,2.554512e+06,diluted,2024-08-14,33.0,3.193140e+06,True
15,AAME,2026-03-06,active,2.6200,2.039700e+07,diluted,2025-11-14,112.0,5.344014e+07,True
16,AAMI,2026-03-06,active,51.1800,3.584662e+07,diluted,2025-11-06,120.0,1.834630e+09,True


In [5]:
## Celda 6: clasificación final

base["classification"] = "unclassified_no_market_cap"

mask_mc = base["market_cap_t"].notna()
mask_lt2b = mask_mc & (base["market_cap_t"] < 2_000_000_000)
mask_ge2b = mask_mc & (base["market_cap_t"] >= 2_000_000_000)
mask_active = base["status_last_row"].eq("active")
mask_inactive = base["status_last_row"].eq("inactive")

base.loc[mask_ge2b, "classification"] = "ge_2b_last_classifiable"
base.loc[mask_lt2b & mask_active, "classification"] = "active_lt_2b_last_classifiable"
base.loc[mask_lt2b & mask_inactive, "classification"] = "inactive_died_lt_2b"

base["classification_reason"] = None
base.loc[base["classification"] == "unclassified_no_market_cap", "classification_reason"] ="no_market_cap_t_at_last_classifiable_date"
base.loc[base["classification"] == "ge_2b_last_classifiable", "classification_reason"] = "market_cap_t_ge_2b"
base.loc[base["classification"] == "active_lt_2b_last_classifiable", "classification_reason"] ="active_and_market_cap_t_lt_2b"
base.loc[base["classification"] == "inactive_died_lt_2b", "classification_reason"] = "inactive_and_market_cap_t_lt_2b"

print("classification_counts:")
print(base["classification"].value_counts(dropna=False).to_string())
print()

print("classification x status_last_row:")
display(pd.crosstab(base["classification"], base["status_last_row"], dropna=False))

print("\nsmallcap_cut:")
smallcap_cut = base.loc[
    base["classification"].isin([
        "active_lt_2b_last_classifiable",
        "inactive_died_lt_2b",
    ])
].copy()

print("smallcap_cut_rows:", len(smallcap_cut))
print("smallcap_cut_unique_tickers:", smallcap_cut["ticker"].nunique())
display(smallcap_cut.head(20))

classification_counts:
classification
active_lt_2b_last_classifiable    5478
unclassified_no_market_cap        4807
ge_2b_last_classifiable           2183

classification x status_last_row:


status_last_row,active
classification,
active_lt_2b_last_classifiable,5478
ge_2b_last_classifiable,2183
unclassified_no_market_cap,4807



smallcap_cut:
smallcap_cut_rows: 5478
smallcap_cut_unique_tickers: 5478


,ticker,first_seen_date,last_observed_date,last_row_date,status_last_row,close_t_last_row,shares_outstanding_t_last_row,shares_source_last_row,shares_observed_date_last_row,shares_period_end_last_row,shares_age_days_last_row,market_cap_t_last_row,is_small_cap_t_last_row,anchor_date_used,status_at_anchor,close_t,shares_outstanding_t,shares_source,shares_observed_date,shares_period_end,shares_age_days,market_cap_t,is_small_cap_t,classification,classification_reason
4,AAC,2008-10-03,2023-11-06,2023-11-06,active,10.7900,NaN,diluted,2023-03-10,2021-12-31,241.0,NaN,None,2023-09-06,active,10.6950,9.375000e+07,diluted,2023-03-10,2021-12-31,180.0,1.002656e+09,True,active_lt_2b_last_classifiable,active_and_market_cap_t_lt_2b
8,AACT,2023-06-12,2025-09-24,2025-09-24,active,9.4900,5.547089e+07,diluted,2025-08-12,2024-06-30,43.0,5.264188e+08,True,2025-09-24,active,9.4900,5.547089e+07,diluted,2025-08-12,2024-06-30,43.0,5.264188e+08,True,active_lt_2b_last_classifiable,active_and_market_cap_t_lt_2b
9,AAGR,2023-12-07,2024-09-25,2024-09-25,active,0.1326,2.839467e+07,diluted,2024-05-20,2024-03-31,128.0,3.765133e+06,True,2024-09-25,active,0.1326,2.839467e+07,diluted,2024-05-20,2024-03-31,128.0,3.765133e+06,True,active_lt_2b_last_classifiable,active_and_market_cap_t_lt_2b
11,AAIC,2020-10-26,2023-12-14,2023-12-14,active,NaN,2.924300e+07,diluted,2023-11-14,2022-12-31,30.0,NaN,None,2023-12-13,active,4.8400,2.924300e+07,diluted,2023-11-14,2022-12-31,29.0,1.415361e+08,True,active_lt_2b_last_classifiable,active_and_market_cap_t_lt_2b
14,AAMC,2014-08-15,2024-09-16,2024-09-16,active,1.2500,2.554512e+06,diluted,2024-08-14,2024-06-30,33.0,3.193140e+06,True,2024-09-16,active,1.2500,2.554512e+06,diluted,2024-08-14,2024-06-30,33.0,3.193140e+06,True,active_lt_2b_last_classifiable,active_and_market_cap_t_lt_2b
15,AAME,2016-10-25,2026-03-09,2026-03-09,active,NaN,2.039700e+07,diluted,2025-11-14,2025-09-30,115.0,NaN,None,2026-03-06,active,2.6200,2.039700e+07,diluted,2025-11-14,2025-09-30,112.0,5.344014e+07,True,active_lt_2b_last_classifiable,active_and_market_cap_t_lt_2b
16,AAMI,2025-01-02,2026-03-09,2026-03-09,active,NaN,3.584662e+07,diluted,2025-11-06,2025-09-30,123.0,NaN,None,2026-03-06,active,51.1800,3.584662e+07,diluted,2025-11-06,2025-09-30,120.0,1.834630e+09,True,active_lt_2b_last_classifiable,active_and_market_cap_t_lt_2b
17,AAN,2009-04-21,2024-10-03,2024-10-03,active,10.0900,3.101925e+07,diluted,2024-08-05,2024-06-30,59.0,3.129842e+08,True,2024-10-03,active,10.0900,3.101925e+07,diluted,2024-08-05,2024-06-30,59.0,3.129842e+08,True,active_lt_2b_last_classifiable,active_and_market_cap_t_lt_2b
23,AAQC,2021-05-10,2022-12-15,2022-12-15,active,NaN,4.780220e+07,diluted,2022-11-07,2022-09-30,38.0,NaN,None,2022-12-14,active,10.1150,4.780220e+07,diluted,2022-11-07,2022-09-30,37.0,4.835192e+08,True,active_lt_2b_last_classifiable,active_and_market_cap_t_lt_2b
24,AARD,2025-02-13,2026-03-09,2026-03-09,active,NaN,1.514646e+07,diluted,2025-11-13,2025-09-30,116.0,NaN,None,2026-03-06,active,5.8500,1.514646e+07,diluted,2025-11-13,2025-09-30,113.0,8.860680e+07,True,active_lt_2b_last_classifiable,active_and_market_cap_t_lt_2b


In [6]:

## Celda 7: auditar por qué no salen inactivas

print("status_last_row_counts:")
print(base["status_last_row"].value_counts(dropna=False).to_string())
print()

print("status_at_anchor_counts:")
print(base["status_at_anchor"].value_counts(dropna=False).to_string())
print()

print("sample_inactive_last_row:")
display(
    base.loc[base["status_last_row"] == "inactive", [
        "ticker",
        "first_seen_date",
        "last_observed_date",
        "last_row_date",
        "status_last_row",
        "anchor_date_used",
        "status_at_anchor",
        "market_cap_t",
        "classification",
    ]].head(30)
)

print("sample_active_last_row:")
display(
    base.loc[base["status_last_row"] == "active", [
        "ticker",
        "first_seen_date",
        "last_observed_date",
        "last_row_date",
        "status_last_row",
        "anchor_date_used",
        "status_at_anchor",
        "market_cap_t",
        "classification",
    ]].head(30)
)

status_last_row_counts:
status_last_row
active    12468

status_at_anchor_counts:
status_at_anchor
active    7661
NaN       4807

sample_inactive_last_row:


,ticker,first_seen_date,last_observed_date,last_row_date,status_last_row,anchor_date_used,status_at_anchor,market_cap_t,classification


sample_active_last_row:


,ticker,first_seen_date,last_observed_date,last_row_date,status_last_row,anchor_date_used,status_at_anchor,market_cap_t,classification
0,A,2005-01-01,2026-03-09,2026-03-09,active,2026-03-06,active,3.267988e+10,ge_2b_last_classifiable
1,AA,2005-01-01,2026-03-09,2026-03-09,active,2026-03-06,active,1.559045e+10,ge_2b_last_classifiable
2,AAA,2005-01-01,2007-05-21,2007-05-21,active,NaT,NaN,NaN,unclassified_no_market_cap
3,AABA,2017-06-19,2019-10-06,2019-10-06,active,NaT,NaN,NaN,unclassified_no_market_cap
4,AAC,2008-10-03,2023-11-06,2023-11-06,active,2023-09-06,active,1.002656e+09,active_lt_2b_last_classifiable
5,AACB,2025-04-07,2026-03-09,2026-03-09,active,NaT,NaN,NaN,unclassified_no_market_cap
6,AACI,2021-11-10,2025-10-29,2025-10-29,active,NaT,NaN,NaN,unclassified_no_market_cap
7,AACQ,2020-09-04,2021-06-24,2021-06-24,active,NaT,NaN,NaN,unclassified_no_market_cap
8,AACT,2023-06-12,2025-09-24,2025-09-24,active,2025-09-24,active,5.264188e+08,active_lt_2b_last_classifiable
9,AAGR,2023-12-07,2024-09-25,2024-09-25,active,2024-09-25,active,3.765133e+06,active_lt_2b_last_classifiable


In [7]:
## Celda 8: reconstruir status operativo por last_observed_date

panel_end_date = base["last_observed_date"].max()
print("panel_end_date:", panel_end_date)

base["status_rebuilt"] = "inactive"
base.loc[base["last_observed_date"] == panel_end_date, "status_rebuilt"] = "active"

print("status_rebuilt_counts:")
print(base["status_rebuilt"].value_counts(dropna=False).to_string())
print()

display(
    base.loc[:, [
        "ticker",
        "first_seen_date",
        "last_observed_date",
        "status_last_row",
        "status_rebuilt",
        "anchor_date_used",
        "market_cap_t",
    ]].head(20)
)

panel_end_date: 2026-03-09 00:00:00
status_rebuilt_counts:
status_rebuilt
inactive    7212
active      5256



,ticker,first_seen_date,last_observed_date,status_last_row,status_rebuilt,anchor_date_used,market_cap_t
0,A,2005-01-01,2026-03-09,active,active,2026-03-06,3.267988e+10
1,AA,2005-01-01,2026-03-09,active,active,2026-03-06,1.559045e+10
2,AAA,2005-01-01,2007-05-21,active,inactive,NaT,NaN
3,AABA,2017-06-19,2019-10-06,active,inactive,NaT,NaN
4,AAC,2008-10-03,2023-11-06,active,inactive,2023-09-06,1.002656e+09
5,AACB,2025-04-07,2026-03-09,active,active,NaT,NaN
6,AACI,2021-11-10,2025-10-29,active,inactive,NaT,NaN
7,AACQ,2020-09-04,2021-06-24,active,inactive,NaT,NaN
8,AACT,2023-06-12,2025-09-24,active,inactive,2025-09-24,5.264188e+08
9,AAGR,2023-12-07,2024-09-25,active,inactive,2024-09-25,3.765133e+06


In [8]:
## Celda 9: clasificación correcta activa/inactiva

base["classification"] = "unclassified_no_market_cap"

mask_mc = base["market_cap_t"].notna()
mask_lt2b = mask_mc & (base["market_cap_t"] < 2_000_000_000)
mask_ge2b = mask_mc & (base["market_cap_t"] >= 2_000_000_000)
mask_active = base["status_rebuilt"].eq("active")
mask_inactive = base["status_rebuilt"].eq("inactive")

base.loc[mask_ge2b, "classification"] = "ge_2b_last_classifiable"
base.loc[mask_lt2b & mask_active, "classification"] = "active_lt_2b_last_classifiable"
base.loc[mask_lt2b & mask_inactive, "classification"] = "inactive_died_lt_2b"

base["classification_reason"] = None
base.loc[base["classification"] == "unclassified_no_market_cap", "classification_reason"] = "no_market_cap_t_at_last_classifiable_date"
base.loc[base["classification"] == "ge_2b_last_classifiable", "classification_reason"] = "market_cap_t_ge_2b"
base.loc[base["classification"] == "active_lt_2b_last_classifiable", "classification_reason"] = "active_and_market_cap_t_lt_2b"
base.loc[base["classification"] == "inactive_died_lt_2b", "classification_reason"] = "inactive_and_market_cap_t_lt_2b"

print("classification_counts:")
print(base["classification"].value_counts(dropna=False).to_string())
print()

print("classification x status_rebuilt:")
display(pd.crosstab(base["classification"], base["status_rebuilt"], dropna=False))

smallcap_cut = base.loc[
    base["classification"].isin([
        "active_lt_2b_last_classifiable",
        "inactive_died_lt_2b",
    ])
].copy()

print()
print("smallcap_cut_rows:", len(smallcap_cut))
print("smallcap_cut_unique_tickers:", smallcap_cut["ticker"].nunique())

display(
    smallcap_cut.loc[:, [
        "ticker",
        "first_seen_date",
        "last_observed_date",
        "status_rebuilt",
        "anchor_date_used",
        "market_cap_t",
        "classification",
    ]].head(20)
)

classification_counts:
classification
unclassified_no_market_cap        4807
active_lt_2b_last_classifiable    2938
inactive_died_lt_2b               2540
ge_2b_last_classifiable           2183

classification x status_rebuilt:


status_rebuilt,active,inactive
classification,,
active_lt_2b_last_classifiable,2938,0
ge_2b_last_classifiable,1736,447
inactive_died_lt_2b,0,2540
unclassified_no_market_cap,582,4225



smallcap_cut_rows: 5478
smallcap_cut_unique_tickers: 5478


,ticker,first_seen_date,last_observed_date,status_rebuilt,anchor_date_used,market_cap_t,classification
4,AAC,2008-10-03,2023-11-06,inactive,2023-09-06,1.002656e+09,inactive_died_lt_2b
8,AACT,2023-06-12,2025-09-24,inactive,2025-09-24,5.264188e+08,inactive_died_lt_2b
9,AAGR,2023-12-07,2024-09-25,inactive,2024-09-25,3.765133e+06,inactive_died_lt_2b
11,AAIC,2020-10-26,2023-12-14,inactive,2023-12-13,1.415361e+08,inactive_died_lt_2b
14,AAMC,2014-08-15,2024-09-16,inactive,2024-09-16,3.193140e+06,inactive_died_lt_2b
15,AAME,2016-10-25,2026-03-09,active,2026-03-06,5.344014e+07,active_lt_2b_last_classifiable
16,AAMI,2025-01-02,2026-03-09,active,2026-03-06,1.834630e+09,active_lt_2b_last_classifiable
17,AAN,2009-04-21,2024-10-03,inactive,2024-10-03,3.129842e+08,inactive_died_lt_2b
23,AAQC,2021-05-10,2022-12-15,inactive,2022-12-14,4.835192e+08,inactive_died_lt_2b
24,AARD,2025-02-13,2026-03-09,active,2026-03-06,8.860680e+07,active_lt_2b_last_classifiable


Has cogido los 12468 tickers del universo y, para cada uno, has intentado mirar su último punto clasificable por
market cap.

“Último punto clasificable” = último día donde pudiste calcular:

`market_cap_t = close_t * shares_outstanding_t`

Qué sale de ahí

**active_lt_2b_last_classifiable = 2938**  
- llegan vivos al final del panel (2026-03-09)
- y en su último punto clasificable estaban < 2B

**inactive_died_lt_2b = 2540**  
- desaparecen antes del final del panel
- y en su último punto clasificable estaban < 2B

**ge_2b_last_classifiable = 2183**  
- sí tienen market cap calculado
- pero en su último punto clasificable estaban >= 2B

Aquí se mezclan:

- activas grandes
- e inactivas que murieron siendo >= 2B

**unclassified_no_market_cap = 4807**

Son tickers para los que no pudiste calcular market cap en el punto final útil.

Normalmente porque faltó alguna de estas piezas:

- close_t
- shares_outstanding_t
- o ambas

**La cuenta total**

Todo suma el universo entero:

- 2938 + 2540 + 2183 + 4807 = 12468

**La idea importante**

De los 12468:

```
- 7661 sí quedaron clasificables por market cap
- 4807 no
```
Y dentro de los 7661 clasificables:
```
- 5478 estaban < 2B
- 2183 estaban >= 2B
```
Y esos 5478 se parten en:
```
- 2938 activas <2B
- 2540 inactivas que murieron <2B
```

**Tu resultado significa:**

- sí puedes construir un corte razonable <2B
- pero solo para 7661 tickers
- y de esos, 5478 encajan en el target small-cap final/interesante


In [9]:
summary_table = pd.DataFrame(
    [
        {
            "group": "active_lt_2b_last_classifiable",
            "tickers": int((base["classification"] == "active_lt_2b_last_classifiable").sum()),
            "meaning": "Tickers que llegan vivos al final del panel y cuyo ultimo market_cap_t clasificable es < 2B",
        },
        {
            "group": "inactive_died_lt_2b",
            "tickers": int((base["classification"] == "inactive_died_lt_2b").sum()),
            "meaning": "Tickers que desaparecen antes del final del panel y cuyo ultimo market_cap_t clasificable es < 2B",
        },
        {
            "group": "ge_2b_last_classifiable",
            "tickers": int((base["classification"] == "ge_2b_last_classifiable").sum()),
            "meaning": "Tickers cuyo ultimo market_cap_t clasificable es >= 2B",
        },
        {
            "group": "unclassified_no_market_cap",
            "tickers": int((base["classification"] == "unclassified_no_market_cap").sum()),
            "meaning": "Tickers sin market_cap_t calculable en el ultimo punto util",
        },
    ]
)

summary_table["pct_universe"] = (summary_table["tickers"] / len(base) * 100).round(2)

print("Resumen del corte por market cap sobre el universo base")
display(summary_table)

print()
print("Chequeos:")
print("universe_tickers:", len(base))
print("classifiable_tickers:", int(base["market_cap_t"].notna().sum()))
print("lt_2b_total:", int((base["market_cap_t"] < 2_000_000_000).sum()))
print("ge_2b_total:", int((base["market_cap_t"] >= 2_000_000_000).sum()))
print("unclassified_total:", int(base["market_cap_t"].isna().sum()))
print()
print("Comprobacion suma clases:", int(summary_table["tickers"].sum()))

Resumen del corte por market cap sobre el universo base


,group,tickers,meaning,pct_universe
0,active_lt_2b_last_classifiable,2938,Tickers que llegan vivos al final del panel y cuyo ultimo market_cap_t clasificable es < 2B,23.56
1,inactive_died_lt_2b,2540,Tickers que desaparecen antes del final del panel y cuyo ultimo market_cap_t clasificable es < 2B,20.37
2,ge_2b_last_classifiable,2183,Tickers cuyo ultimo market_cap_t clasificable es >= 2B,17.51
3,unclassified_no_market_cap,4807,Tickers sin market_cap_t calculable en el ultimo punto util,38.55



Chequeos:
universe_tickers: 12468
classifiable_tickers: 7661
lt_2b_total: 5478
ge_2b_total: 2183
unclassified_total: 4807

Comprobacion suma clases: 12468


In [10]:
## Celda 10: guardar artefactos

OUT_DIR.mkdir(parents=True, exist_ok=True)

detail_cols = [
    "ticker",
    "first_seen_date",
    "last_observed_date",
    "status_rebuilt",
    "last_row_date",
    "anchor_date_used",
    "close_t",
    "shares_outstanding_t",
    "shares_source",
    "shares_observed_date",
    "shares_period_end",
    "shares_age_days",
    "market_cap_t",
    "is_small_cap_t",
    "classification",
    "classification_reason",
]

detail = base[detail_cols].copy()
smallcap_cut = base.loc[
    base["classification"].isin([
        "active_lt_2b_last_classifiable",
        "inactive_died_lt_2b",
    ]),
    detail_cols
].copy()

detail.to_parquet(DETAIL_PARQUET, index=False)
detail.to_csv(DETAIL_CSV, index=False)

smallcap_cut.to_parquet(CUT_PARQUET, index=False)
smallcap_cut.to_csv(CUT_CSV, index=False)

summary = {
    "run_id": RUN_ID,
    "panel_end_date": str(panel_end_date.date()),
    "universe_tickers": int(base["ticker"].nunique()),
    "status_rebuilt_counts": {k: int(v) for k, v in
base["status_rebuilt"].value_counts(dropna=False).to_dict().items()},
    "classification_counts": {k: int(v) for k, v in
base["classification"].value_counts(dropna=False).to_dict().items()},
    "outputs": {
        "detail_parquet": str(DETAIL_PARQUET),
        "detail_csv": str(DETAIL_CSV),
        "cut_parquet": str(CUT_PARQUET),
        "cut_csv": str(CUT_CSV),
    },
}

SUMMARY_JSON.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

print("saved:")
print(DETAIL_PARQUET)
print(DETAIL_CSV)
print(CUT_PARQUET)
print(CUT_CSV)
print(SUMMARY_JSON)

saved:
C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\market_cap_last_observed_cutoff\20260320_market_cap_last_observed_cutoff\market_cap_last_observed_by_ticker.parquet
C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\market_cap_last_observed_cutoff\20260320_market_cap_last_observed_cutoff\market_cap_last_observed_by_ticker.csv
C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\market_cap_last_observed_cutoff\20260320_market_cap_last_observed_cutoff\market_cap_cutoff_lt_2b_active_inactive.parquet
C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\market_cap_last_observed_cutoff\20260320_market_cap_last_observed_cutoff\market_cap_cutoff_lt_2b_active_inactive.csv
C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\market_cap_last_observed_cutoff\20260320_market_cap_last_observed_cutoff\market_cap_cutoff_lt_2b_summary.json


In [11]:
## Celda: definir cutoff 1B

CUTOFF_B = 1_000_000_000
CUTOFF_LABEL = "1b"

DETAIL_PARQUET_1B = OUT_DIR / f"market_cap_last_observed_by_ticker_{CUTOFF_LABEL}.parquet"
DETAIL_CSV_1B = OUT_DIR / f"market_cap_last_observed_by_ticker_{CUTOFF_LABEL}.csv"
CUT_PARQUET_1B = OUT_DIR / f"market_cap_cutoff_lt_{CUTOFF_LABEL}_active_inactive.parquet"
CUT_CSV_1B = OUT_DIR / f"market_cap_cutoff_lt_{CUTOFF_LABEL}_active_inactive.csv"
SUMMARY_JSON_1B = OUT_DIR / f"market_cap_cutoff_lt_{CUTOFF_LABEL}_summary.json"

print("CUTOFF_B:", CUTOFF_B)
print("CUT_PARQUET_1B:", CUT_PARQUET_1B)

## Celda: clasificación 1B

base["classification_1b"] = "unclassified_no_market_cap"

mask_mc = base["market_cap_t"].notna()
mask_lt1b = mask_mc & (base["market_cap_t"] < CUTOFF_B)
mask_ge1b = mask_mc & (base["market_cap_t"] >= CUTOFF_B)
mask_active = base["status_rebuilt"].eq("active")
mask_inactive = base["status_rebuilt"].eq("inactive")

base.loc[mask_ge1b, "classification_1b"] = "ge_1b_last_classifiable"
base.loc[mask_lt1b & mask_active, "classification_1b"] = "active_lt_1b_last_classifiable"
base.loc[mask_lt1b & mask_inactive, "classification_1b"] = "inactive_died_lt_1b"

base["classification_reason_1b"] = None
base.loc[base["classification_1b"] == "unclassified_no_market_cap", "classification_reason_1b"] ="no_market_cap_t_at_last_classifiable_date"
base.loc[base["classification_1b"] == "ge_1b_last_classifiable", "classification_reason_1b"] = "market_cap_t_ge_1b"
base.loc[base["classification_1b"] == "active_lt_1b_last_classifiable", "classification_reason_1b"] ="active_and_market_cap_t_lt_1b"
base.loc[base["classification_1b"] == "inactive_died_lt_1b", "classification_reason_1b"] ="inactive_and_market_cap_t_lt_1b"

print("classification_1b_counts:")
print(base["classification_1b"].value_counts(dropna=False).to_string())
print()

print("classification_1b x status_rebuilt:")
display(pd.crosstab(base["classification_1b"], base["status_rebuilt"], dropna=False))

## Celda: tabla resumen 1B

summary_table_1b = pd.DataFrame(
    [
        {
            "group": "active_lt_1b_last_classifiable",
            "tickers": int((base["classification_1b"] == "active_lt_1b_last_classifiable").sum()),
            "meaning": "Tickers activos al final del panel con ultimo market_cap_t clasificable < 1B",
        },
        {
            "group": "inactive_died_lt_1b",
            "tickers": int((base["classification_1b"] == "inactive_died_lt_1b").sum()),
            "meaning": "Tickers inactivos antes del final del panel con ultimo market_cap_t clasificable < 1B",
        },
        {
            "group": "ge_1b_last_classifiable",
            "tickers": int((base["classification_1b"] == "ge_1b_last_classifiable").sum()),
            "meaning": "Tickers cuyo ultimo market_cap_t clasificable es >= 1B",
        },
        {
            "group": "unclassified_no_market_cap",
            "tickers": int((base["classification_1b"] == "unclassified_no_market_cap").sum()),
            "meaning": "Tickers sin market_cap_t calculable en el ultimo punto util",
        },
    ]
)

summary_table_1b["pct_universe"] = (summary_table_1b["tickers"] / len(base) * 100).round(2)

display(summary_table_1b)

print("Comprobacion suma clases:", int(summary_table_1b["tickers"].sum()))

## Celda: generar artefactos 1B

OUT_DIR.mkdir(parents=True, exist_ok=True)

detail_cols_1b = [
    "ticker",
    "first_seen_date",
    "last_observed_date",
    "status_rebuilt",
    "last_row_date",
    "anchor_date_used",
    "close_t",
    "shares_outstanding_t",
    "shares_source",
    "shares_observed_date",
    "shares_period_end",
    "shares_age_days",
    "market_cap_t",
    "is_small_cap_t",
    "classification_1b",
    "classification_reason_1b",
]

detail_1b = base[detail_cols_1b].copy()
cut_1b = base.loc[
    base["classification_1b"].isin([
        "active_lt_1b_last_classifiable",
        "inactive_died_lt_1b",
    ]),
    detail_cols_1b
].copy()

detail_1b.to_parquet(DETAIL_PARQUET_1B, index=False)
detail_1b.to_csv(DETAIL_CSV_1B, index=False)

cut_1b.to_parquet(CUT_PARQUET_1B, index=False)
cut_1b.to_csv(CUT_CSV_1B, index=False)

summary_1b = {
    "run_id": RUN_ID,
    "cutoff": 1_000_000_000,
    "panel_end_date": str(panel_end_date.date()),
    "universe_tickers": int(base["ticker"].nunique()),
    "status_rebuilt_counts": {k: int(v) for k, v in
base["status_rebuilt"].value_counts(dropna=False).to_dict().items()},
    "classification_counts": {k: int(v) for k, v in
base["classification_1b"].value_counts(dropna=False).to_dict().items()},
    "outputs": {
        "detail_parquet": str(DETAIL_PARQUET_1B),
        "detail_csv": str(DETAIL_CSV_1B),
        "cut_parquet": str(CUT_PARQUET_1B),
        "cut_csv": str(CUT_CSV_1B),
    },
}

SUMMARY_JSON_1B.write_text(json.dumps(summary_1b, indent=2, ensure_ascii=False), encoding="utf-8")

print("saved:")
print(DETAIL_PARQUET_1B)
print(DETAIL_CSV_1B)
print(CUT_PARQUET_1B)
print(CUT_CSV_1B)
print(SUMMARY_JSON_1B)
print()
print("cut_1b_rows:", len(cut_1b))
print("cut_1b_unique_tickers:", cut_1b["ticker"].nunique())

CUTOFF_B: 1000000000
CUT_PARQUET_1B: C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\market_cap_last_observed_cutoff\20260320_market_cap_last_observed_cutoff\market_cap_cutoff_lt_1b_active_inactive.parquet
classification_1b_counts:
classification_1b
unclassified_no_market_cap        4807
ge_1b_last_classifiable           2837
active_lt_1b_last_classifiable    2476
inactive_died_lt_1b               2348

classification_1b x status_rebuilt:


status_rebuilt,active,inactive
classification_1b,,
active_lt_1b_last_classifiable,2476,0
ge_1b_last_classifiable,2198,639
inactive_died_lt_1b,0,2348
unclassified_no_market_cap,582,4225


,group,tickers,meaning,pct_universe
0,active_lt_1b_last_classifiable,2476,Tickers activos al final del panel con ultimo market_cap_t clasificable < 1B,19.86
1,inactive_died_lt_1b,2348,Tickers inactivos antes del final del panel con ultimo market_cap_t clasificable < 1B,18.83
2,ge_1b_last_classifiable,2837,Tickers cuyo ultimo market_cap_t clasificable es >= 1B,22.75
3,unclassified_no_market_cap,4807,Tickers sin market_cap_t calculable en el ultimo punto util,38.55


Comprobacion suma clases: 12468
saved:
C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\market_cap_last_observed_cutoff\20260320_market_cap_last_observed_cutoff\market_cap_last_observed_by_ticker_1b.parquet
C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\market_cap_last_observed_cutoff\20260320_market_cap_last_observed_cutoff\market_cap_last_observed_by_ticker_1b.csv
C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\market_cap_last_observed_cutoff\20260320_market_cap_last_observed_cutoff\market_cap_cutoff_lt_1b_active_inactive.parquet
C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\market_cap_last_observed_cutoff\20260320_market_cap_last_observed_cutoff\market_cap_cutoff_lt_1b_active_inactive.csv
C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\market_cap_last_observed_cutoff\20260320_market_cap_last_observed_cutoff\market_cap_cutoff_lt_1b_summary.json

cut_1b_rows: 4824
cut_1b_unique_tickers: 4824


Comparación del recovery actual de quotes contra el nuevo corte `<1B`

In [14]:
from pathlib import Path
import pandas as pd

quotes_tasks_path = Path(
    r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260313_quotes_prod_full_12133_clean\inputs\tasks_quotes_prod_missing_only_final_v2.csv"
)

cut_1b_path = Path(
    r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\market_cap_last_observed_cutoff\20260320_market_cap_last_observed_cutoff\market_cap_cutoff_lt_1b_active_inactive.parquet"
)

tasks = pd.read_csv(quotes_tasks_path, usecols=["ticker"]).copy()
tasks["ticker"] = tasks["ticker"].astype(str).str.upper().str.strip()
tasks = tasks.dropna().drop_duplicates().sort_values("ticker").reset_index(drop=True)

cut = pd.read_parquet(cut_1b_path).copy()
cut["ticker"] = cut["ticker"].astype(str).str.upper().str.strip()
cut = cut.dropna(subset=["ticker"]).drop_duplicates(subset=["ticker"]).sort_values("ticker").reset_index(drop=True)

quotes_set = set(tasks["ticker"])
cut_set = set(cut["ticker"])

inside = sorted(quotes_set & cut_set)
outside = sorted(quotes_set - cut_set)
only_cut = sorted(cut_set - quotes_set)

print("quotes_tasks_path:", quotes_tasks_path)
print("cut_1b_path:", cut_1b_path)
print()
print("quotes_unique_tickers:", len(quotes_set))
print("cut_1b_unique_tickers:", len(cut_set))
print("quotes_inside_cut_1b:", len(inside))
print("quotes_outside_cut_1b:", len(outside))
print("only_cut_1b_not_in_quotes:", len(only_cut))
print()
print("pct_quotes_inside_cut_1b:", round(len(inside) / len(quotes_set) * 100, 2) if quotes_set else None)

comparison = pd.DataFrame({
    "ticker": sorted(quotes_set | cut_set)
})
comparison["in_quotes_tasks"] = comparison["ticker"].isin(quotes_set)
comparison["in_cut_1b"] = comparison["ticker"].isin(cut_set)

print("\nSample quotes outside cut_1b:")
display(comparison.loc[(comparison["in_quotes_tasks"]) & (~comparison["in_cut_1b"])].head(5))

print("\nSample quotes inside cut_1b:")
display(comparison.loc[(comparison["in_quotes_tasks"]) & (comparison["in_cut_1b"])].head(5))

quotes_tasks_path: C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260313_quotes_prod_full_12133_clean\inputs\tasks_quotes_prod_missing_only_final_v2.csv
cut_1b_path: C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\market_cap_last_observed_cutoff\20260320_market_cap_last_observed_cutoff\market_cap_cutoff_lt_1b_active_inactive.parquet

quotes_unique_tickers: 1582
cut_1b_unique_tickers: 4824
quotes_inside_cut_1b: 1235
quotes_outside_cut_1b: 347
only_cut_1b_not_in_quotes: 3589

pct_quotes_inside_cut_1b: 78.07

Sample quotes outside cut_1b:


,ticker,in_quotes_tasks,in_cut_1b
0,AACI,True,False
9,AAT,True,False
13,ABCL,True,False
28,ABVE,True,False
37,ACCL,True,False



Sample quotes inside cut_1b:


,ticker,in_quotes_tasks,in_cut_1b
12,ABAT,True,True
18,ABLV,True,True
19,ABOS,True,True
21,ABSI,True,True
27,ABVC,True,True


In [72]:
from pathlib import Path
import pandas as pd

p = Path(
    r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\market_cap_last_observed_cutoff\20260320_market_cap_last_observed_cutoff\market_cap_cutoff_lt_1b_active_inactive.parquet"
)

df = pd.read_parquet(p)

print("path:", p)
print("shape:", df.shape)
df.head(1).T

path: C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\market_cap_last_observed_cutoff\20260320_market_cap_last_observed_cutoff\market_cap_cutoff_lt_1b_active_inactive.parquet
shape: (4824, 16)


,0
ticker,AACT
first_seen_date,2023-06-12 00:00:00
last_observed_date,2025-09-24 00:00:00
status_rebuilt,inactive
last_row_date,2025-09-24 00:00:00
anchor_date_used,2025-09-24 00:00:00
close_t,9.49
shares_outstanding_t,55470890.5
shares_source,diluted
shares_observed_date,2025-08-12 00:00:00


Comparación del recovery actual de quotes contra el nuevo corte `<1B`

Se comparó:
```
C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260313_quotes_prod_full_12133_clean\inputs\tasks_quotes
_prod_missing_only_final_v2.csv
```
contra:
```
C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\market_cap_last_observed_cutoff\20260320_market_cap_last_observed_cutof
f\market_cap_cutoff_lt_1b_active_inactive.parquet
```
Resultado:

- quotes_unique_tickers: 1582
- cut_1b_unique_tickers: 4824
- quotes_inside_cut_1b: 1235
- quotes_outside_cut_1b: 347
- pct_quotes_inside_cut_1b: 78.07%

Interpretación:

- el 78.07% del recovery actual de quotes cae dentro del nuevo universo `<1B`
- el 21.93% del recovery actual de quotes quedaría fuera si se aplica el nuevo cutoff
- por tanto, el nuevo corte `<1B` sí reduce de forma material la descarga sin vaciar el universo operativo

In [17]:
from pathlib import Path
import pandas as pd

quotes_universe_path = Path(
    r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260313_quotes_prod_full_12133_clean\inputs\tickers_quotes_prod.csv"
)

cut_1b_path = Path(
    r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\market_cap_last_observed_cutoff\20260320_market_cap_last_observed_cutoff\market_cap_cutoff_lt_1b_active_inactive.parquet"
)

quotes = pd.read_csv(quotes_universe_path, usecols=["ticker"]).copy()
quotes["ticker"] = quotes["ticker"].astype(str).str.upper().str.strip()
quotes = quotes.dropna().drop_duplicates().sort_values("ticker").reset_index(drop=True)

cut = pd.read_parquet(cut_1b_path).copy()
cut["ticker"] = cut["ticker"].astype(str).str.upper().str.strip()
cut = cut.dropna(subset=["ticker"]).drop_duplicates(subset=["ticker"]).sort_values("ticker").reset_index(drop=True)

quotes_set = set(quotes["ticker"])
cut_set = set(cut["ticker"])

inside = sorted(quotes_set & cut_set)
outside = sorted(quotes_set - cut_set)
only_cut = sorted(cut_set - quotes_set)

print("quotes_universe_path:", quotes_universe_path)
print("cut_1b_path:", cut_1b_path)
print()
print("quotes_unique_tickers:", len(quotes_set))
print("cut_1b_unique_tickers:", len(cut_set))
print("quotes_inside_cut_1b:", len(inside))
print("quotes_outside_cut_1b:", len(outside))
print("only_cut_1b_not_in_quotes:", len(only_cut))
print()
print("pct_quotes_inside_cut_1b:", round(len(inside) / len(quotes_set) * 100, 2) if quotes_set else None)

comparison = pd.DataFrame({
    "ticker": sorted(quotes_set | cut_set)
})
comparison["in_quotes_universe"] = comparison["ticker"].isin(quotes_set)
comparison["in_cut_1b"] = comparison["ticker"].isin(cut_set)

print("\nSample quotes outside cut_1b:")
display(comparison.loc[(comparison["in_quotes_universe"]) & (~comparison["in_cut_1b"])].head(5))

print("\nSample quotes inside cut_1b:")
display(comparison.loc[(comparison["in_quotes_universe"]) & (comparison["in_cut_1b"])].head(5))

quotes_universe_path: C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260313_quotes_prod_full_12133_clean\inputs\tickers_quotes_prod.csv
cut_1b_path: C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\market_cap_last_observed_cutoff\20260320_market_cap_last_observed_cutoff\market_cap_cutoff_lt_1b_active_inactive.parquet

quotes_unique_tickers: 1961
cut_1b_unique_tickers: 4824
quotes_inside_cut_1b: 1544
quotes_outside_cut_1b: 417
only_cut_1b_not_in_quotes: 3280

pct_quotes_inside_cut_1b: 78.74

Sample quotes outside cut_1b:


,ticker,in_quotes_universe,in_cut_1b
0,AABA,True,False
1,AACI,True,False
10,AAT,True,False
14,ABCL,True,False
22,ABR,True,False



Sample quotes inside cut_1b:


,ticker,in_quotes_universe,in_cut_1b
13,ABAT,True,True
15,ABEO,True,True
19,ABLV,True,True
20,ABOS,True,True
21,ABP,True,True


In [19]:
from pathlib import Path
import pandas as pd

trades_universe_path = Path(
    r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\trades_ticks_prod_2005_2026\inputs\tickers_trades_ticks.csv"
)

cut_1b_path = Path(
    r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\market_cap_last_observed_cutoff\20260320_market_cap_last_observed_cutoff\market_cap_cutoff_lt_1b_active_inactive.parquet"
)

trades = pd.read_csv(trades_universe_path, usecols=["ticker"]).copy()
trades["ticker"] = trades["ticker"].astype(str).str.upper().str.strip()
trades = trades.dropna().drop_duplicates().sort_values("ticker").reset_index(drop=True)

cut = pd.read_parquet(cut_1b_path).copy()
cut["ticker"] = cut["ticker"].astype(str).str.upper().str.strip()
cut = cut.dropna(subset=["ticker"]).drop_duplicates(subset=["ticker"]).sort_values("ticker").reset_index(drop=True)

trades_set = set(trades["ticker"])
cut_set = set(cut["ticker"])

inside = sorted(trades_set & cut_set)
outside = sorted(trades_set - cut_set)
only_cut = sorted(cut_set - trades_set)

print("trades_universe_path:", trades_universe_path)
print("cut_1b_path:", cut_1b_path)
print()
print("trades_unique_tickers:", len(trades_set))
print("cut_1b_unique_tickers:", len(cut_set))
print("trades_inside_cut_1b:", len(inside))
print("trades_outside_cut_1b:", len(outside))
print("only_cut_1b_not_in_trades:", len(only_cut))
print()
print("pct_trades_inside_cut_1b:", round(len(inside) / len(trades_set) * 100, 2) if trades_set else None)

comparison = pd.DataFrame({
    "ticker": sorted(trades_set | cut_set)
})
comparison["in_trades_universe"] = comparison["ticker"].isin(trades_set)
comparison["in_cut_1b"] = comparison["ticker"].isin(cut_set)

print("\nSample trades outside cut_1b:")
display(comparison.loc[(comparison["in_trades_universe"]) & (~comparison["in_cut_1b"])].head(5))

print("\nSample trades inside cut_1b:")
display(comparison.loc[(comparison["in_trades_universe"]) & (comparison["in_cut_1b"])].head(5))

trades_universe_path: C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\trades_ticks_prod_2005_2026\inputs\tickers_trades_ticks.csv
cut_1b_path: C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\market_cap_last_observed_cutoff\20260320_market_cap_last_observed_cutoff\market_cap_cutoff_lt_1b_active_inactive.parquet

trades_unique_tickers: 1962
cut_1b_unique_tickers: 4824
trades_inside_cut_1b: 1544
trades_outside_cut_1b: 418
only_cut_1b_not_in_trades: 3280

pct_trades_inside_cut_1b: 78.7

Sample trades outside cut_1b:


,ticker,in_trades_universe,in_cut_1b
0,AABA,True,False
1,AACI,True,False
10,AAT,True,False
14,ABCL,True,False
22,ABR,True,False



Sample trades inside cut_1b:


,ticker,in_trades_universe,in_cut_1b
13,ABAT,True,True
15,ABEO,True,True
19,ABLV,True,True
20,ABOS,True,True
21,ABP,True,True


In [20]:
## Celda: construir universo operativo <1B solo con ticker

from pathlib import Path
import pandas as pd

cut_1b_path = Path(
    r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\market_cap_last_observed_cutoff\20260320_market_cap_last_observed_cutoff\market_cap_cutoff_lt_1b_active_inactive.parquet"
)

out_parquet = Path(
    r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_2005_2026_upper_lt_1b_last_observed.parquet"
)

out_csv = Path(
    r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_2005_2026_upper_lt_1b_last_observed.csv"
)

d = pd.read_parquet(cut_1b_path).copy()
d["ticker"] = d["ticker"].astype(str).str.upper().str.strip()

u_1b = (
    d[["ticker"]]
    .dropna()
    .drop_duplicates()
    .sort_values("ticker")
    .reset_index(drop=True)
)

u_1b.to_parquet(out_parquet, index=False)
u_1b.to_csv(out_csv, index=False)

print("cut_1b_path:", cut_1b_path)
print("out_parquet:", out_parquet)
print("out_csv:", out_csv)
print()
print("tickers_resultado:", len(u_1b))
display(u_1b.head(20))

cut_1b_path: C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\market_cap_last_observed_cutoff\20260320_market_cap_last_observed_cutoff\market_cap_cutoff_lt_1b_active_inactive.parquet
out_parquet: C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_2005_2026_upper_lt_1b_last_observed.parquet
out_csv: C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_2005_2026_upper_lt_1b_last_observed.csv

tickers_resultado: 4824


,ticker
0,AACT
1,AAGR
2,AAIC
3,AAMC
4,AAME
5,AAN
6,AAQC
7,AARD
8,AATC
9,ABAC


## Inventario actual de tiks & quotes descargados 21/03/2026

```sh
# quotes
powershell 
  -NoProfile 
  -ExecutionPolicy Bypass 
  -File C:\Users\AlexJ\inventory_scripts\inventory_quotes_final_files.ps1 -ProgressEverySec 5 
  -OutTxt C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260313_quotes_prod_full_12133_clean\inputs\quotes_final_file_paths_v2.txt

=== quotes inventory ===
root=D:\quotes
out=C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260313_quotes_prod_full_12133_clean\inputs\quotes_final_file_paths_v2.txt
mode=enumeratefiles_stream
DONE [quotes] lines=2204172 elapsed=9.933,5s out_mb=119,45

# trades tiks
powershell 
  -NoProfile 
  -ExecutionPolicy Bypass 
  -File C:\Users\AlexJ\inventory_scripts\inventory_trades_ticks_final_files.ps1 
  -ProgressEverySec 5

=== trades_ticks inventory ===
root=D:\trades_ticks_prod_2005_2026
out=C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\trades_ticks_prod_2005_2026\inputs\trades_ticks_final_file_paths.txt
mode=enumeratefiles_stream
DONE [trades_ticks] lines=1205751 elapsed=9.279,3s out_mb=98,69
```

In [27]:
from pathlib import Path
import pandas as pd

txt_path = Path(
    r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260313_quotes_prod_full_12133_clean\inputs\quotes_final_file_paths_v2.txt"
)

out_csv = txt_path.with_name(txt_path.stem + "_summary_by_ticker.csv")
out_parquet = txt_path.with_name(txt_path.stem + "_summary_by_ticker.parquet")

df = pd.read_csv(txt_path, header=None, names=["file_path"])
df["file_path"] = df["file_path"].astype(str).str.strip()

parts = df["file_path"].str.split("\\", expand=True)

# Esperado para quotes:
# D:\quotes\TICKER\year=YYYY\month=MM\day=DD\quotes.parquet
df["ticker"] = parts[2].astype(str).str.upper().str.strip()
df["year"] = pd.to_numeric(parts[3].str.replace("year=", "", regex=False), errors="coerce")
df["month"] = pd.to_numeric(parts[4].str.replace("month=", "", regex=False), errors="coerce")
df["day"] = pd.to_numeric(parts[5].str.replace("day=", "", regex=False), errors="coerce")

df["date"] = pd.to_datetime(
    dict(year=df["year"], month=df["month"], day=df["day"]),
    errors="coerce"
)

summary = (
    df.groupby("ticker", as_index=False)
    .agg(
        files_count=("file_path", "size"),
        date_min=("date", "min"),
        date_max=("date", "max"),
    )
    .sort_values(["files_count", "ticker"], ascending=[False, True])
    .reset_index(drop=True)
)

summary.to_csv(out_csv, index=False)
summary.to_parquet(out_parquet, index=False)

print("txt_path:", txt_path)
print("out_csv:", out_csv)
print("out_parquet:", out_parquet)
print()
print("files_total:", len(df))
print("tickers_total:", summary["ticker"].nunique())
print()

display(summary.head(5))

txt_path: C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260313_quotes_prod_full_12133_clean\inputs\quotes_final_file_paths_v2.txt
out_csv: C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260313_quotes_prod_full_12133_clean\inputs\quotes_final_file_paths_v2_summary_by_ticker.csv
out_parquet: C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260313_quotes_prod_full_12133_clean\inputs\quotes_final_file_paths_v2_summary_by_ticker.parquet

files_total: 2204172
tickers_total: 1731



,ticker,files_count,date_min,date_max
0,ASYS,5663,2003-09-10,2026-03-13
1,AXTI,5663,2003-09-10,2026-03-13
2,BELFB,5663,2003-09-10,2026-03-13
3,BYFC,5663,2003-09-10,2026-03-13
4,CLWT,5663,2003-09-10,2026-03-13


In [28]:
## Siguiente celda para trades_ticks

from pathlib import Path
import pandas as pd

txt_path = Path(
    r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\trades_ticks_prod_2005_2026\inputs\trades_ticks_final_file_paths.txt"
)

out_csv = txt_path.with_name(txt_path.stem + "_summary_by_ticker.csv")
out_parquet = txt_path.with_name(txt_path.stem + "_summary_by_ticker.parquet")

df = pd.read_csv(txt_path, header=None, names=["file_path"])
df["file_path"] = df["file_path"].astype(str).str.strip()

parts = df["file_path"].str.split("\\", expand=True)

# Esperado para trades_ticks:
# D:\trades_ticks_prod_2005_2026\TICKER\year=YYYY\month=MM\day=YYYY-MM-DD\market.parquet
df["ticker"] = parts[2].astype(str).str.upper().str.strip()
df["year"] = pd.to_numeric(parts[3].str.replace("year=", "", regex=False), errors="coerce")
df["month"] = pd.to_numeric(parts[4].str.replace("month=", "", regex=False), errors="coerce")
df["day_str"] = parts[5].str.replace("day=", "", regex=False)

df["date"] = pd.to_datetime(df["day_str"], errors="coerce")

summary = (
    df.groupby("ticker", as_index=False)
    .agg(
        files_count=("file_path", "size"),
        date_min=("date", "min"),
        date_max=("date", "max"),
    )
    .sort_values(["files_count", "ticker"], ascending=[False, True])
    .reset_index(drop=True)
)

summary.to_csv(out_csv, index=False)
summary.to_parquet(out_parquet, index=False)

print("txt_path:", txt_path)
print("out_csv:", out_csv)
print("out_parquet:", out_parquet)
print()
print("files_total:", len(df))
print("tickers_total:", summary["ticker"].nunique())
print()

display(summary.head(5))

txt_path: C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\trades_ticks_prod_2005_2026\inputs\trades_ticks_final_file_paths.txt
out_csv: C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\trades_ticks_prod_2005_2026\inputs\trades_ticks_final_file_paths_summary_by_ticker.csv
out_parquet: C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\trades_ticks_prod_2005_2026\inputs\trades_ticks_final_file_paths_summary_by_ticker.parquet

files_total: 1205751
tickers_total: 932



,ticker,files_count,date_min,date_max
0,AXTI,5661,2003-09-10,2026-03-13
1,BELFB,5661,2003-09-10,2026-03-13
2,CNTY,5661,2003-09-10,2026-03-13
3,ASYS,5657,2003-09-10,2026-03-13
4,ISSC,5645,2003-09-10,2026-03-13


In [36]:
from pathlib import Path
import pandas as pd

path = Path(
    r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_2005_2026_upper_excluding_ohlcv_1m_missing_vs_daily_ge_2B_or_null.parquet"
)

df = pd.read_parquet(path).copy()

print("path:", path)
print("shape:", df.shape)
print("columns:", list(df.columns))
print()

if "ticker" in df.columns:
    df["ticker"] = df["ticker"].astype(str).str.upper().str.strip()
    print("tickers_unique:", df["ticker"].dropna().nunique())
    print()

path: C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_2005_2026_upper_excluding_ohlcv_1m_missing_vs_daily_ge_2B_or_null.parquet
shape: (12133, 1)
columns: ['ticker']

tickers_unique: 12133



In [50]:
from pathlib import Path
import pandas as pd

root = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_panel_pti")

files = sorted(root.rglob("*.parquet"))

print("root:", root)
print("n_files:", len(files))
print()

for p in files[:50]:
    print(p)

# opcional: guardar listado completo
out_csv = root.parent / "tickers_panel_pti_all_parquets.csv"
pd.DataFrame({"parquet_path": [str(p) for p in files]}).to_csv(out_csv, index=False)

print()
print("saved_list_csv:", out_csv)

root: C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_panel_pti
n_files: 0


saved_list_csv: C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_panel_pti_all_parquets.csv


In [51]:
from pathlib import Path

root = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_panel_pti")
files = sorted(root.rglob("*.parquet"))

print("n_files:", len(files))
for p in files:
    print(p)

n_files: 0


In [38]:
## Celda 1

from pathlib import Path

UNIVERSE_REFINED = Path(
    r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_2005_2026_upper_excluding_ohlcv_1m_missing_vs_daily_ge_2B_or_null.parquet"
)

POP_RUN_DIR = Path(
    r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\population_target_pti\population_target_pti_run_01"
)

POP_PARQUET = POP_RUN_DIR / "population_target_pti.parquet"
POP_SUMMARY = POP_RUN_DIR / "population_target_pti_summary.json"
POP_YEAR_SUMMARY = POP_RUN_DIR / "population_target_pti_year_summary.parquet"

NOTEBOOK = Path(
    r"C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\00_data_certification\00_descarga_universo.ipynb"
)

SCRIPT = Path(
    r"C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\build_population_target_pti.py"
)

print("UNIVERSE_REFINED:", UNIVERSE_REFINED)
print("POP_PARQUET:", POP_PARQUET)
print("POP_SUMMARY:", POP_SUMMARY)
print("POP_YEAR_SUMMARY:", POP_YEAR_SUMMARY)
print("NOTEBOOK:", NOTEBOOK)
print("SCRIPT:", SCRIPT)


UNIVERSE_REFINED: C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_2005_2026_upper_excluding_ohlcv_1m_missing_vs_daily_ge_2B_or_null.parquet
POP_PARQUET: C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\population_target_pti\population_target_pti_run_01\population_target_pti.parquet
POP_SUMMARY: C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\population_target_pti\population_target_pti_run_01\population_target_pti_summary.json
POP_YEAR_SUMMARY: C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\population_target_pti\population_target_pti_run_01\population_target_pti_year_summary.parquet
NOTEBOOK: C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\00_data_certification\00_descarga_universo.ipynb
SCRIPT: C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\build_population_target_pti.py


In [ ]:

## Celda 2
import pandas as pd

u_ref = pd.read_parquet(UNIVERSE_REFINED).copy()
u_ref["ticker"] = u_ref["ticker"].astype(str).str.upper().str.strip()

print("UNIVERSE_REFINED")
print("shape:", u_ref.shape)
print("tickers_unique:", u_ref["ticker"].nunique())
print("columns:", list(u_ref.columns))
display(u_ref.head(1))

UNIVERSE_REFINED
shape: (12133, 1)
tickers_unique: 12133
columns: ['ticker']


,ticker
0,AAA


In [42]:

## Celda 3
import json

summary = json.loads(POP_SUMMARY.read_text(encoding="utf-8"))

print("population_target_pti_summary.json")
print(json.dumps(summary, indent=2, ensure_ascii=False))


population_target_pti_summary.json
{
  "created_at_utc": "2026-03-10T19:02:25.853845+00:00",
  "date_from": "2005-01-01",
  "date_to": "2026-12-31",
  "ttl_days": 180,
  "max_tickers": null,
  "paths_used": {
    "universe_pti_root": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\data\\reference\\universe_pti\\tickers_panel_pti",
    "ohlcv_daily_root": "D:\\ohlcv_daily",
    "income_statements_root": "D:\\financial\\income_statements",
    "balance_sheets_root": "D:\\financial\\balance_sheets",
    "output_parquet": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\runs\\backtest\\population_target_pti\\population_target_pti_run_01\\population_target_pti.parquet",
    "year_summary_parquet": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\runs\\backtest\\population_target_pti\\population_target_pti_run_01\\population_target_pti_year_summary.parquet"
  },
  "metrics": {
    "rows_total": 29735570,
    "tickers_total": 13066,
    "rows_with_close_t": 19702631,
    "rows_with_shares_t": 15143484,
    "rows_class

In [43]:

## Celda 4

print("paths_used:")
for k, v in summary["paths_used"].items():
    print(f"- {k}: {v}")

print()
print("metrics:")
for k, v in summary["metrics"].items():
    print(f"- {k}: {v}")



paths_used:
- universe_pti_root: C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_panel_pti
- ohlcv_daily_root: D:\ohlcv_daily
- income_statements_root: D:\financial\income_statements
- balance_sheets_root: D:\financial\balance_sheets
- output_parquet: C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\population_target_pti\population_target_pti_run_01\population_target_pti.parquet
- year_summary_parquet: C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\population_target_pti\population_target_pti_run_01\population_target_pti_year_summary.parquet

metrics:
- rows_total: 29735570
- tickers_total: 13066
- rows_with_close_t: 19702631
- rows_with_shares_t: 15143484
- rows_classifiable: 10278625
- rows_small_cap: 6071914
- anti_lookahead_violations: 0


In [44]:
## Celda 5

df_pop = pd.read_parquet(POP_PARQUET)

print("population_target_pti.parquet")
print("shape:", df_pop.shape)
print("columns:", list(df_pop.columns))
display(df_pop.head(5).T)



population_target_pti.parquet
shape: (29735570, 19)
columns: ['date', 'ticker', 'entity_id', 'figi_like', 'primary_exchange', 'status', 'close_t', 'shares_outstanding_t', 'shares_source', 'shares_observed_date', 'shares_period_end', 'shares_age_days', 'market_cap_t', 'is_small_cap_t', 'year', 'month', 'volume_t', 'vwap_t', 'trades_t']


,0,1,2,3,4
date,2016-10-25,2016-10-26,2016-10-27,2016-10-28,2016-10-29
ticker,AAWW,AAWW,AAWW,AAWW,AAWW
entity_id,BBG000Q57YP0,BBG000Q57YP0,BBG000Q57YP0,BBG000Q57YP0,BBG000Q57YP0
figi_like,BBG000Q57YP0,BBG000Q57YP0,BBG000Q57YP0,BBG000Q57YP0,BBG000Q57YP0
primary_exchange,XNAS,XNAS,XNAS,XNAS,XNAS
status,active,active,active,active,active
close_t,42.15,42.35,42.05,42.15,NaN
shares_outstanding_t,25198000.0,25198000.0,25198000.0,25198000.0,25198000.0
shares_source,diluted,diluted,diluted,diluted,diluted
shares_observed_date,2016-08-03,2016-08-03,2016-08-03,2016-08-03,2016-08-03


In [45]:
## Celda 6

year_summary = pd.read_parquet(POP_YEAR_SUMMARY)

print("population_target_pti_year_summary.parquet")
print("shape:", year_summary.shape)
display(year_summary)

population_target_pti_year_summary.parquet
shape: (22, 7)


,year,rows_total,rows_with_close_t,rows_with_shares_t,rows_classifiable,rows_small_cap,anti_lookahead_violations
0,2005,964990,646932,0,0,0.0,0.0
1,2006,958653,640894,0,0,0.0,0.0
2,2007,946452,633567,0,0,0.0,0.0
3,2008,991323,662410,0,0,0.0,0.0
4,2009,1117483,735267,0,0,0.0,0.0
5,2010,1087812,726115,15962,11130,2922.0,0.0
6,2011,1088287,728124,233510,160510,64949.0,0.0
7,2012,1095261,726100,407316,274760,134388.0,0.0
8,2013,1093427,735205,438393,299806,138213.0,0.0
9,2014,1114691,750878,465892,319465,140512.0,0.0


In [46]:
## Celda 7

print("INPUTS reales usados por build_population_target_pti.py")
print("-", summary["paths_used"]["universe_pti_root"])
print("-", summary["paths_used"]["ohlcv_daily_root"])
print("-", summary["paths_used"]["income_statements_root"])
print("-", summary["paths_used"]["balance_sheets_root"])


INPUTS reales usados por build_population_target_pti.py
- C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_panel_pti
- D:\ohlcv_daily
- D:\financial\income_statements
- D:\financial\balance_sheets


In [47]:
## Celda 8

script_text = SCRIPT.read_text(encoding="utf-8")

markers = [
    "UNIVERSE_PTI_ROOT",
    "OHLCV_DAILY_ROOT",
    "INCOME_STATEMENTS_ROOT",
    "BALANCE_SHEETS_ROOT",
    "CREATE OR REPLACE TEMP VIEW spine_base AS",
    "CREATE OR REPLACE TEMP VIEW daily_close AS",
    "CREATE OR REPLACE TEMP VIEW shares_obs AS",
    "ASOF LEFT JOIN shares_obs",
    "market_cap_t",
    "is_small_cap_t",
    "TTL_DAYS",
]

for m in markers:
    print("=" * 80)
    print(m)
    idx = script_text.find(m)
    if idx >= 0:
        start = max(0, idx - 250)
        end = min(len(script_text), idx + 900)
        print(script_text[start:end])
    else:
        print("NO ENCONTRADO")

UNIVERSE_PTI_ROOT
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import duckdb
import pandas as pd


UNIVERSE_PTI_ROOT = Path(
    globals().get(
        "UNIVERSE_PTI_ROOT",
        r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_panel_pti",
    )
)
OHLCV_DAILY_ROOT = Path(globals().get("OHLCV_DAILY_ROOT", r"D:\ohlcv_daily"))
INCOME_STATEMENTS_ROOT = Path(globals().get("INCOME_STATEMENTS_ROOT", r"D:\financial\income_statements"))
BALANCE_SHEETS_ROOT = Path(globals().get("BALANCE_SHEETS_ROOT", r"D:\financial\balance_sheets"))

RUN_ID = str(globals().get("RUN_ID", datetime.now().strftime("%Y%m%d_%H%M%S_population_target_pti")))
OUT_BASE = Path(
    globals().get(
        "OUT_BASE",
        r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\population_target_pti",
    )
)
OUT_DIR = Path(globals().get("OUT_DIR", OUT_BASE / RUN_ID))
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATE_FROM = str(global

In [48]:
## Celda 9

print("Reconstrucción conceptual del script")
print()
print("1. Lee el spine PTI diario desde:")
print("   ", summary["paths_used"]["universe_pti_root"])
print()
print("2. Lee close_t desde:")
print("   ", summary["paths_used"]["ohlcv_daily_root"])
print()
print("3. Lee shares_outstanding observadas desde:")
print("   ", summary["paths_used"]["income_statements_root"])
print()
print("4. Hace ASOF LEFT JOIN backward por filing_date")
print("5. Aplica TTL_DAYS = ", summary["ttl_days"])
print("6. Calcula market_cap_t = close_t * shares_outstanding_t")
print("7. Calcula is_small_cap_t = market_cap_t < 2_000_000_000")
print()
print("Output principal:")
print("   ", summary["paths_used"]["output_parquet"])

## Celda 10

print("Encaje en el hilo")
print()
print("1. Primero se construyó el universo refinado candidato:")
print("   ", UNIVERSE_REFINED)
print()
print("2. Luego, en paralelo, se construyó el artefacto PTI global de market cap:")
print("   ", POP_PARQUET)
print()
print("3. Ese artefacto PTI no redefinió automáticamente el universo operativo de quotes/trades_ticks.")
print("4. Sí sirvió después para construir cortes globales por market cap en 00_corte_2B_universo.ipynb.")

Reconstrucción conceptual del script

1. Lee el spine PTI diario desde:
    C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_panel_pti

2. Lee close_t desde:
    D:\ohlcv_daily

3. Lee shares_outstanding observadas desde:
    D:\financial\income_statements

4. Hace ASOF LEFT JOIN backward por filing_date
5. Aplica TTL_DAYS =  180
6. Calcula market_cap_t = close_t * shares_outstanding_t
7. Calcula is_small_cap_t = market_cap_t < 2_000_000_000

Output principal:
    C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\population_target_pti\population_target_pti_run_01\population_target_pti.parquet
Encaje en el hilo

1. Primero se construyó el universo refinado candidato:
    C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_2005_2026_upper_excluding_ohlcv_1m_missing_vs_daily_ge_2B_or_null.parquet

2. Luego, en paralelo, se construyó el artefacto PTI global de market cap:
    C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\population_target_pti\popu